# App Logs Analysis
This notebook loads the `app_logs.txt` file, parses the space-separated values into proper columns, and registers it as a temporary SQL table so you can query it!

In [ ]:
import os
import sys

# Ensure the notebook can find PySpark in our virtual environment FIRST
venv_site_packages = os.path.join(os.getcwd(), 'venv', 'Lib', 'site-packages')
if venv_site_packages not in sys.path:
    sys.path.insert(0, venv_site_packages)

# Set HADOOP_HOME and JAVA_HOME
current_dir = os.getcwd()
os.environ['HADOOP_HOME'] = os.path.join(current_dir, 'hadoop')
os.environ['JAVA_HOME'] = os.path.join(current_dir, 'jdk-17')

# NOW we can import pyspark safely
from pyspark.sql import SparkSession
from pyspark.sql.functions import split
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("LogAnalysisDF").master("local[*]").getOrCreate()

# Load data
df = spark.read.text("Sep_1/app_logs.txt")

# Split into columns
split_col = split(df["value"], " ")

logs = df.select(
    split_col.getItem(0).alias("date"),
    split_col.getItem(1).alias("time"),
    split_col.getItem(2).alias("level"),
    split_col.getItem(3).alias("user"),
    split_col.getItem(4).alias("action"),
    split_col.getItem(5).alias("status_or_item")
)

# Register as temp table for SQL
logs.createOrReplaceTempView("logs")

print("\u2705 Logs successfully parsed and registered as the 'logs' table!")
logs.show(10, truncate=False)

✅ Logs successfully parsed and registered as the 'logs' table!
+----------+--------+-----+-----+---------------+--------------+
|date      |time    |level|user |action         |status_or_item|
+----------+--------+-----+-----+---------------+--------------+
|2024-01-01|10:00:01|INFO |user1|login          |success       |
|2024-01-01|10:01:15|ERROR|user2|payment        |failed        |
|2024-01-01|10:02:20|INFO |user3|view_product   |item123       |
|2024-01-01|10:03:05|WARN |user1|low_balance    |NULL          |
|2024-01-01|10:04:45|INFO |user4|add_to_cart    |item456       |
|2024-01-01|10:05:12|ERROR|user2|payment        |failed        |
|2024-01-01|10:06:30|INFO |user5|logout         |success       |
|2024-01-01|10:07:55|INFO |user3|purchase       |item123       |
|2024-01-01|10:08:10|ERROR|user6|login          |failed        |
|2024-01-01|10:09:25|INFO |user1|purchase       |item789       |
|2024-01-01|10:10:40|WARN |user4|session_timeout|NULL          |
|2024-01-01|10:11:22|INFO |

### Write SQL Queries Below
Now that the `logs` table is ready, you can query it normally.

In [ ]:
query = """
SELECT level, COUNT(*) as count
FROM logs
GROUP BY level
ORDER BY count DESC
"""

spark.sql(query).show()

+-----+-----+
|level|count|
+-----+-----+
| INFO|   12|
|ERROR|    5|
| WARN|    3|
+-----+-----+



In [ ]:
no_purchase_query = """
SELECT DISTINCT user 
FROM logs 
WHERE user NOT IN (
    SELECT DISTINCT user 
    FROM logs 
    WHERE action = 'purchase'
)
ORDER BY user
"""

spark.sql(no_purchase_query).show()


+-----+
| user|
+-----+
|user2|
|user4|
|user5|
|user6|
+-----+



In [ ]:
cleaner_duration_query = """
WITH user_times AS (
    SELECT 
        user,
        MIN(CASE WHEN action = 'login' THEN time END) as login_time,
        MAX(CASE WHEN action = 'logout' THEN time END) as logout_time
    FROM logs
    GROUP BY user
)
SELECT 
    user,
    login_time,
    logout_time,
    -- Tell Spark the string is just Hours:Minutes:Seconds
    unix_timestamp(logout_time, 'HH:mm:ss') - unix_timestamp(login_time, 'HH:mm:ss') AS total_seconds_active
FROM user_times
WHERE login_time IS NOT NULL AND logout_time IS NOT NULL
ORDER BY total_seconds_active DESC
"""

spark.sql(cleaner_duration_query).show()


+-----+----------+-----------+--------------------+
| user|login_time|logout_time|total_seconds_active|
+-----+----------+-----------+--------------------+
|user1|  10:00:01|   10:16:29|                 988|
|user5|  10:19:40|   10:06:30|                -790|
+-----+----------+-----------+--------------------+



In [ ]:
active_users_query = """
SELECT 
    user, 
    COUNT(*) AS total_actions
FROM logs
GROUP BY user
ORDER BY total_actions DESC
"""

spark.sql(active_users_query).show()


+-----+-------------+
| user|total_actions|
+-----+-------------+
|user1|            4|
|user3|            4|
|user2|            4|
|user5|            3|
|user4|            3|
|user6|            2|
+-----+-------------+



In [ ]:
intersect_query = """
SELECT DISTINCT user FROM logs WHERE status_or_item = 'success'
INTERSECT
SELECT DISTINCT user FROM logs WHERE status_or_item = 'failed'
"""

spark.sql(intersect_query).show()


+-----+
| user|
+-----+
|user2|
+-----+



In [ ]:
failed_logins_query = """
SELECT 
    user, 
    COUNT(*) as failed_login_attempts
FROM logs
WHERE action = 'login' AND status_or_item = 'failed'
GROUP BY user
ORDER BY failed_login_attempts DESC
"""

spark.sql(failed_logins_query).show()


+-----+---------------------+
| user|failed_login_attempts|
+-----+---------------------+
|user4|                    1|
|user6|                    1|
+-----+---------------------+



In [ ]:
consecutive_failures_query = """
WITH PreviousStatus AS (
    SELECT 
        user,
        time,
        action,
        status_or_item,
        LAG(status_or_item) OVER (PARTITION BY user ORDER BY time) as previous_status
    FROM logs
)
SELECT 
    user,
    time,
    action
FROM PreviousStatus
WHERE status_or_item = 'failed' AND previous_status = 'failed'
"""

spark.sql(consecutive_failures_query).show()


+-----+--------+-------+
| user|    time| action|
+-----+--------+-------+
|user2|10:05:12|payment|
+-----+--------+-------+

